# 보이스피싱 전사 데이터 검수·정량평가

이 노트북은 `cases.json`, `turns.csv`, `검수필요.csv`를 읽어 층화표본 50개와 검수 채점표를 생성합니다. 사람이 채점표를 작성한 뒤 다시 업로드하면 CER/WER, 역할 분류 지표, 사건 경계 지표, 발화 단위 화자 일치율, 변조 음성별 성능과 Markdown 보고서를 자동 생성합니다.

주의: 기존 결과에는 pyannote 원시 구간 타임라인이 저장되지 않았으므로 정식 DER 대신 **발화 시간 가중 화자 일치율**을 계산합니다. 정식 DER은 `DER_정밀구간` 시트에 사람 정답 구간과 자동 RTTM 구간이 모두 있을 때만 별도 계산해야 합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pandas openpyxl jiwer scikit-learn scipy matplotlib seaborn
from pathlib import Path
import json, math, re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

## 1. 경로 설정

아래 두 경로만 실제 Drive 위치에 맞게 고칩니다. `분석 결과` 아래에는 세 분류의 `cases.json`이, `원본 영상 및 음원` 아래에는 원본 MP3/MP4가 있어야 합니다.

In [ ]:
DRIVE = Path('/content/drive/MyDrive')
RESULT_ROOT = DRIVE / '보이스피싱_분석' / '분석 결과'
AUDIO_ROOT = DRIVE / '보이스피싱_분석' / '원본 영상 및 음원'
REPORT_ROOT = DRIVE / '보이스피싱_분석' / '검수 및 보고서'
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
N_SAMPLE = 50
RANDOM_SEED = 20260812
BOUNDARY_TOLERANCE_SEC = 2.0
print('결과 JSON:', len(list(RESULT_ROOT.rglob('cases.json'))))
print('원본 미디어:', len([p for p in AUDIO_ROOT.rglob('*') if p.suffix.lower() in {'.mp3','.mp4'}]))
assert RESULT_ROOT.exists(), f'결과 경로를 확인하세요: {RESULT_ROOT}'

## 2. 결과 통합 및 50개 층화표본 선정

분류, 단일/다중 사건, 검수필요 비율, 길이 구간을 함께 고려합니다. 각 층에서 최소 1개를 우선 뽑고 남은 수는 층 크기에 비례해 고정 난수로 선정합니다. 같은 시드에서는 같은 표본이 재현됩니다.

In [ ]:
def category_from(path, source_file):
    text = str(source_file) + ' ' + str(path)
    for name in ['바로 이 목소리', '그놈 목소리 - 대출사기형', '그놈 목소리 - 수사기관형']:
        if name in text: return name
    return path.relative_to(RESULT_ROOT).parts[0] if path.is_relative_to(RESULT_ROOT) else '기타'

file_rows, turn_rows, case_rows = [], [], []
for jp in RESULT_ROOT.rglob('cases.json'):
    try:
        d = json.loads(jp.read_text(encoding='utf-8'))
        source = d.get('source_file', '')
        category = category_from(jp, source)
        all_turns = [t for c in d.get('cases', []) for t in c.get('turns', [])]
        duration = float(d.get('metadata', {}).get('duration') or max([t.get('end',0) for t in all_turns] or [0]))
        review_count = sum(1 for t in all_turns if not t.get('role') or float(t.get('role_confidence') or 0) < 0.6)
        file_id = jp.parent.name
        file_rows.append({'file_id':file_id,'category':category,'source_file':source,'json_path':str(jp),'duration_sec':duration,'case_count':len(d.get('cases',[])),'turn_count':len(all_turns),'review_count':review_count,'review_ratio':review_count/max(1,len(all_turns))})
        for c in d.get('cases', []):
            case_rows.append({'file_id':file_id,'category':category,'source_file':source,'case_id':c.get('case_id'),'auto_case_start':c.get('start'),'auto_case_end':c.get('end'),'boundary_reason':c.get('boundary_reason'),'auto_needs_review':c.get('needs_review')})
            for i,t in enumerate(c.get('turns',[]),1):
                turn_rows.append({'file_id':file_id,'category':category,'source_file':source,'case_id':c.get('case_id'),'turn_id':i,'start':t.get('start'),'end':t.get('end'),'duration_sec':max(0,float(t.get('end',0))-float(t.get('start',0))),'auto_speaker_id':t.get('speaker_id',''),'auto_role':t.get('role',''),'role_heuristic_score':t.get('role_confidence',0),'auto_text':t.get('text',''),'avg_logprob':t.get('avg_logprob')})
    except Exception as e:
        print('JSON 오류:', jp, e)
files_df = pd.DataFrame(file_rows).drop_duplicates('file_id')
turns_df = pd.DataFrame(turn_rows)
cases_df = pd.DataFrame(case_rows)
assert len(files_df), 'cases.json을 찾지 못했습니다.'
files_df['multi_case'] = np.where(files_df.case_count > 1, '다중', '단일')
files_df['review_band'] = pd.cut(files_df.review_ratio, [-.001,.05,.20,1.001], labels=['낮음','중간','높음'])
files_df['length_band'] = pd.qcut(files_df.duration_sec.rank(method='first'), q=min(3,len(files_df)), labels=['짧음','중간','김'], duplicates='drop')
files_df['stratum'] = files_df.category.astype(str)+'|'+files_df.multi_case.astype(str)+'|'+files_df.review_band.astype(str)+'|'+files_df.length_band.astype(str)

def stratified_sample(df, n, seed):
    rng = np.random.default_rng(seed); groups = [g for _,g in df.groupby('stratum', observed=True)]
    selected=[]
    if len(groups) > n:
        groups = sorted(groups, key=len, reverse=True)[:n]
    for g in groups:
        selected.extend(rng.choice(g.index.to_numpy(), size=1, replace=False).tolist())
    selected=list(dict.fromkeys(selected))
    remaining=n-len(selected)
    pool=df.drop(index=selected)
    if remaining>0 and len(pool):
        weights=pool.groupby('stratum',observed=True)['file_id'].transform('count').astype(float)
        selected.extend(rng.choice(pool.index.to_numpy(),size=min(remaining,len(pool)),replace=False,p=(weights/weights.sum()).to_numpy()).tolist())
    return df.loc[selected].sort_values(['category','file_id']).reset_index(drop=True)
sample_df = stratified_sample(files_df, min(N_SAMPLE,len(files_df)), RANDOM_SEED)
sample_df.insert(0,'sample_no',range(1,len(sample_df)+1))
sample_ids=set(sample_df.file_id)
sample_turns=turns_df[turns_df.file_id.isin(sample_ids)].merge(sample_df[['file_id','sample_no']],on='file_id',how='left').sort_values(['sample_no','case_id','start'])
sample_cases=cases_df[cases_df.file_id.isin(sample_ids)].merge(sample_df[['file_id','sample_no']],on='file_id',how='left').sort_values(['sample_no','auto_case_start'])
print('전체 파일:',len(files_df),'표본:',len(sample_df),'표본 발화:',len(sample_turns),'표본 사건:',len(sample_cases))
display(sample_df.groupby(['category','multi_case','review_band'],observed=True).size().rename('표본수').reset_index())

## 3. 검수 채점표 생성

`정답_텍스트`, `정답_화자ID`, `정답_역할`, `음성변조`, 사건 경계를 사람이 입력합니다. 자동값은 수정하지 않습니다. 판단 불가는 `UNSURE`, 학습 제외는 `EXCLUDE`를 사용합니다.

In [ ]:
score_path = REPORT_ROOT / '검수_채점표.xlsx'
guide = pd.DataFrame([
 ['검수자','검수자 이름 또는 ID'],['검수상태','미검수 / 검수중 / 완료 / 제외'],['정답_텍스트','음성을 듣고 정확한 대사를 입력'],['정답_화자ID','사건 안에서 일관된 PERSON_01, PERSON_02 등'],['정답_역할','OFFENDER / VICTIM / THIRD_PARTY / UNSURE / EXCLUDE'],['음성변조','TRUE / FALSE / UNSURE'],['정답_사건시작·종료','초 단위. 자동 경계가 맞아도 값을 복사해 확정'],['경계판정','CORRECT / ADJUSTED / MERGE / SPLIT / UNSURE'],['주의','개인정보는 보고서 본문에 복사하지 않으며 판단 불가를 억지로 확정하지 않음']
],columns=['항목','입력 방법'])
turn_review=sample_turns.copy()
for col,default in [('검수자',''),('검수상태','미검수'),('정답_텍스트',''),('정답_화자ID',''),('정답_역할',''),('음성변조',''),('검수메모','')]: turn_review[col]=default
case_review=sample_cases.copy()
for col,default in [('검수자',''),('검수상태','미검수'),('정답_사건시작',np.nan),('정답_사건종료',np.nan),('경계판정',''),('음성변조포함',''),('검수메모','')]: case_review[col]=default
der_columns=['sample_no','file_id','source_file','case_id','정답_구간시작','정답_구간종료','정답_화자ID','정답_역할','음성변조','검수자','메모']
der_template=pd.DataFrame(columns=der_columns)
with pd.ExcelWriter(score_path,engine='openpyxl') as writer:
    guide.to_excel(writer,sheet_name='안내',index=False)
    sample_df.to_excel(writer,sheet_name='표본파일',index=False)
    case_review.to_excel(writer,sheet_name='사건검수',index=False)
    turn_review.to_excel(writer,sheet_name='발화검수',index=False)
    der_template.to_excel(writer,sheet_name='DER_정밀구간',index=False)
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
wb=load_workbook(score_path)
for ws in wb.worksheets:
    ws.freeze_panes='A2'; ws.auto_filter.ref=ws.dimensions
    for c in ws[1]: c.font=Font(bold=True,color='FFFFFF'); c.fill=PatternFill('solid',fgColor='1F4E78'); c.alignment=Alignment(horizontal='center')
    for col in ws.columns:
        letter=col[0].column_letter; width=min(45,max(10,max(len(str(x.value or '')) for x in col[:200])+2)); ws.column_dimensions[letter].width=width
    ws.sheet_view.showGridLines=False
def add_list(ws,header,values):
    headers={c.value:c.column_letter for c in ws[1]}
    if header in headers:
        dv=DataValidation(type='list',formula1='"'+','.join(values)+'"',allow_blank=True); ws.add_data_validation(dv); dv.add(f'{headers[header]}2:{headers[header]}1048576')
for wsname in ['발화검수','사건검수']:
    add_list(wb[wsname],'검수상태',['미검수','검수중','완료','제외'])
add_list(wb['발화검수'],'정답_역할',['OFFENDER','VICTIM','THIRD_PARTY','UNSURE','EXCLUDE'])
add_list(wb['발화검수'],'음성변조',['TRUE','FALSE','UNSURE'])
add_list(wb['사건검수'],'경계판정',['CORRECT','ADJUSTED','MERGE','SPLIT','UNSURE'])
add_list(wb['사건검수'],'음성변조포함',['TRUE','FALSE','UNSURE'])
wb.save(score_path)
print('생성 완료:',score_path)

## 4. Colab에서 원본 재생 및 자동 결과 확인

`show_sample(1)`처럼 표본 번호를 입력합니다. 원본 경로가 다르면 `AUDIO_ROOT`를 수정합니다. 재생하면서 Drive의 채점표를 Excel 또는 Google Sheets로 열어 정답 칸을 입력합니다.

In [ ]:
from IPython.display import Audio, display, Markdown
media_index={p.name:p for p in AUDIO_ROOT.rglob('*') if p.suffix.lower() in {'.mp3','.mp4'}} if AUDIO_ROOT.exists() else {}
def find_audio(source_file):
    p=AUDIO_ROOT/source_file
    if p.exists(): return p
    return media_index.get(Path(source_file).name)
def show_sample(sample_no):
    row=sample_df[sample_df.sample_no==sample_no].iloc[0]
    audio=find_audio(row.source_file)
    display(Markdown(f"### {sample_no}. {row['category']} — `{row['source_file']}`"))
    print('사건후보:',row.case_count,'발화:',row.turn_count,'검수필요비율:',f'{row.review_ratio:.1%}')
    if audio: display(Audio(filename=str(audio)))
    else: print('원본을 찾지 못했습니다. AUDIO_ROOT 확인:',AUDIO_ROOT)
    display(sample_turns[sample_turns.sample_no==sample_no][['case_id','start','end','auto_speaker_id','auto_role','role_heuristic_score','auto_text']])
show_sample(1)

## 5. 검수 완료 후 정량평가

채점표의 `발화검수`, `사건검수`를 저장한 후 실행합니다. 최소 조건은 발화·사건 행의 `검수상태=완료`입니다. `role_heuristic_score`는 확률이 아니며 보고서에서는 규칙점수로 표기합니다.

In [ ]:
from jiwer import cer, wer
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
from scipy.optimize import linear_sum_assignment
tr=pd.read_excel(score_path,sheet_name='발화검수')
cr=pd.read_excel(score_path,sheet_name='사건검수')
tr=tr[tr['검수상태'].eq('완료')].copy(); cr=cr[cr['검수상태'].eq('완료')].copy()
assert len(tr)>0, '발화검수에서 완료된 행이 없습니다.'
text_eval=tr[tr['정답_텍스트'].fillna('').str.strip().ne('')].copy()
tr['auto_role_eval']=tr['auto_role'].where(tr['auto_role'].isin(['OFFENDER','VICTIM']),'REVIEW')
tr['음성변조_eval']=tr['음성변조'].astype(str).str.upper().replace({'1':'TRUE','0':'FALSE'})
role_eval=tr[tr['정답_역할'].isin(['OFFENDER','VICTIM'])].copy()
overall_cer=cer(text_eval['정답_텍스트'].astype(str).tolist(),text_eval['auto_text'].astype(str).tolist()) if len(text_eval) else np.nan
overall_wer=wer(text_eval['정답_텍스트'].astype(str).tolist(),text_eval['auto_text'].astype(str).tolist()) if len(text_eval) else np.nan
labels=['OFFENDER','VICTIM']
report=classification_report(role_eval['정답_역할'],role_eval['auto_role_eval'],labels=labels,output_dict=True,zero_division=0) if len(role_eval) else {}
cm_labels=['OFFENDER','VICTIM','REVIEW']
cm=confusion_matrix(role_eval['정답_역할'],role_eval['auto_role_eval'],labels=cm_labels) if len(role_eval) else np.zeros((3,3),dtype=int)
role_accuracy=accuracy_score(role_eval['정답_역할'],role_eval['auto_role_eval']) if len(role_eval) else np.nan

def speaker_weighted_accuracy(df):
    good=df[df['정답_화자ID'].fillna('').str.strip().ne('') & df['auto_speaker_id'].fillna('').str.strip().ne('')].copy()
    correct=total=0.0
    for _,g in good.groupby(['file_id','case_id']):
        auto=sorted(g.auto_speaker_id.unique()); gold=sorted(g['정답_화자ID'].unique())
        mat=np.zeros((len(auto),len(gold)))
        for _,r in g.iterrows(): mat[auto.index(r.auto_speaker_id),gold.index(r['정답_화자ID'])]+=max(.001,float(r.duration_sec))
        rr,cc=linear_sum_assignment(-mat); correct+=mat[rr,cc].sum(); total+=mat.sum()
    return correct/total if total else np.nan
speaker_acc=speaker_weighted_accuracy(tr)

def boundary_metrics(df,tol):
    d=df.dropna(subset=['정답_사건시작','정답_사건종료']).copy()
    if not len(d): return {'start_within_tolerance':np.nan,'end_within_tolerance':np.nan,'both_within_tolerance':np.nan,'mean_start_abs_error_sec':np.nan,'mean_end_abs_error_sec':np.nan,'n':0}
    se=(d.auto_case_start-d['정답_사건시작']).abs(); ee=(d.auto_case_end-d['정답_사건종료']).abs()
    return {'start_within_tolerance':float((se<=tol).mean()),'end_within_tolerance':float((ee<=tol).mean()),'both_within_tolerance':float(((se<=tol)&(ee<=tol)).mean()),'mean_start_abs_error_sec':float(se.mean()),'mean_end_abs_error_sec':float(ee.mean()),'n':len(d)}
boundary=boundary_metrics(cr,BOUNDARY_TOLERANCE_SEC)

modified=[]
for value,g in tr[tr['음성변조_eval'].isin(['TRUE','FALSE'])].groupby('음성변조_eval'):
    te=g[g['정답_텍스트'].fillna('').str.strip().ne('')]
    re_=g[g['정답_역할'].isin(labels)]
    modified.append({'음성변조':value,'발화수':len(g),'CER':cer(te['정답_텍스트'].astype(str).tolist(),te.auto_text.astype(str).tolist()) if len(te) else np.nan,'WER':wer(te['정답_텍스트'].astype(str).tolist(),te.auto_text.astype(str).tolist()) if len(te) else np.nan,'역할정확도':accuracy_score(re_['정답_역할'],re_.auto_role_eval) if len(re_) else np.nan})
modified_df=pd.DataFrame(modified)
metrics={'검수파일수':int(tr.file_id.nunique()),'검수발화수':len(tr),'전사평가발화수':len(text_eval),'역할평가발화수':len(role_eval),'CER':overall_cer,'WER':overall_wer,'역할정확도':role_accuracy,'역할MacroF1':report.get('macro avg',{}).get('f1-score',np.nan),'발화시간가중_화자일치율':speaker_acc,'사건경계':boundary}
print(json.dumps(metrics,ensure_ascii=False,indent=2))
display(pd.DataFrame(cm,index=['정답_OFFENDER','정답_VICTIM','정답_REVIEW'],columns=['자동_OFFENDER','자동_VICTIM','자동_REVIEW']))
display(modified_df)

## 6. 보고서·표·혼동행렬 자동 저장

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
plt.figure(figsize=(6,5)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=cm_labels,yticklabels=cm_labels); plt.xlabel('자동 역할'); plt.ylabel('사람 정답'); plt.title('역할 분류 혼동행렬'); plt.tight_layout(); cm_path=REPORT_ROOT/'역할_혼동행렬.png'; plt.savefig(cm_path,dpi=160); plt.show()
pd.DataFrame([metrics | {'사건경계':json.dumps(boundary,ensure_ascii=False)}]).to_csv(REPORT_ROOT/'정량평가_요약.csv',index=False,encoding='utf-8-sig')
modified_df.to_csv(REPORT_ROOT/'변조음성별_성능.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(report).T.to_csv(REPORT_ROOT/'역할분류_상세지표.csv',encoding='utf-8-sig')
auto_summary={'입력결과파일':len(files_df),'자동사건후보':int(files_df.case_count.sum()),'전체발화':len(turns_df),'자동학습후보':int(((turns_df.auto_role.isin(labels))&(turns_df.role_heuristic_score>=.6)).sum()),'자동검수필요':int((~turns_df.auto_role.isin(labels)|(turns_df.role_heuristic_score<.6)).sum())}
report_md=f'''# 보이스피싱 전사 데이터 정량적 신뢰성 검증 보고서

## 1. 데이터 구축 현황

- 처리 결과 파일: {auto_summary['입력결과파일']:,}개
- 자동 사건 후보: {auto_summary['자동사건후보']:,}개
- 전체 발화: {auto_summary['전체발화']:,}개
- 규칙점수 기준 학습 후보: {auto_summary['자동학습후보']:,}개
- 자동 검수 필요: {auto_summary['자동검수필요']:,}개

## 2. 검증 설계

분류, 단일/다중 사건, 자동 검수필요 비율, 길이 구간을 고려한 층화표본 {len(sample_df)}개를 선정했다. 실제 검수 완료 파일은 {metrics['검수파일수']}개, 발화는 {metrics['검수발화수']:,}개이다. 표본 난수 시드는 {RANDOM_SEED}이다.

## 3. 정량 결과

- CER: {overall_cer:.4f}
- WER: {overall_wer:.4f}
- 역할 정확도: {role_accuracy:.4f}
- 역할 Macro F1: {metrics['역할MacroF1']:.4f}
- 발화 시간 가중 화자 일치율: {speaker_acc:.4f}
- 사건 시작·종료 모두 ±{BOUNDARY_TOLERANCE_SEC:.1f}초 이내: {boundary['both_within_tolerance']:.4f}
- 사건 시작 평균 절대오차: {boundary['mean_start_abs_error_sec']:.3f}초
- 사건 종료 평균 절대오차: {boundary['mean_end_abs_error_sec']:.3f}초

## 4. 해석상 주의

`role_heuristic_score`는 통계적으로 보정된 확률이 아니라 키워드 규칙점수다. 기존 `cases.json`에는 pyannote의 원시 화자 구간 타임라인이 없어 정식 DER을 계산하지 않았으며, 대신 사건별 최적 화자 매핑 후 발화 지속시간으로 가중한 화자 일치율을 제시했다. 자동 사건 수는 후보 수이며 사람 검수 전 확정 사건 수가 아니다. 결과는 검수 표본에서 측정된 성능이며 전체 데이터의 무오류를 의미하지 않는다. 개인정보가 포함될 수 있으므로 보고서에는 원문 발화를 직접 인용하지 않는다.

## 5. 권고

ML 학습에는 사람 검수 완료 데이터와 낮은 오류율이 확인된 자동 후보를 구분해 사용하고, 미검수 자동 라벨은 별도 실험군으로 관리한다. 변조 음성 집단의 표본 수가 충분한지 확인하고 집단별 성능 차이가 크면 별도 모델 또는 데이터 증강을 검토한다.
'''
report_path=REPORT_ROOT/'정량적_신뢰성_검증_보고서.md'; report_path.write_text(report_md,encoding='utf-8')
build_md=f'''# 보이스피싱 데이터 구축·자동처리 보고서

- 분석 결과 파일: {auto_summary['입력결과파일']:,}개
- 자동 사건 후보: {auto_summary['자동사건후보']:,}개
- 전체 발화: {auto_summary['전체발화']:,}개
- 자동 학습 후보: {auto_summary['자동학습후보']:,}개
- 자동 검수 필요: {auto_summary['자동검수필요']:,}개

전사는 `mobiuslabsgmbh/faster-whisper-large-v3-turbo`, 화자분리는 `pyannote/speaker-diarization-community-1`을 사용했다. 범인·피해자 역할 및 사건 경계는 규칙 기반 후보이며 사람 검수 전 확정 라벨이 아니다.
'''
(REPORT_ROOT/'데이터_구축_자동처리_보고서.md').write_text(build_md,encoding='utf-8')
print('저장 위치:',REPORT_ROOT)
for p in sorted(REPORT_ROOT.iterdir()): print('-',p.name)